In [ ]:
import math
import tqdm
import random

# import feature_generators
# import timeseries

import pandas as pd
import numpy as np
import scipy as sp

from collections import defaultdict
from datetime import datetime
from itertools import product

from matplotlib import pyplot as plt
%matplotlib inline

from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split
from sklearn.multioutput import MultiOutputClassifier, MultiOutputRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer, mean_squared_error
from sklearn.svm import LinearSVC, SVC

# np.random.seed(42)
# random.seed(a=472443)

# From http://s.arboreus.com/2009/04/cyrillic-letters-in-matplotlibpylab.html
# from matplotlib import rc
# rc('font',**{'family':'serif'})
# rc('text', usetex=True)
# rc('text.latex',unicode=True)
# rc('text.latex',preamble=r'\usepackage[utf8]{inputenc}')
# rc('text.latex',preamble=r'\usepackage[russian]{babel}')

import pcst_fast

In [ ]:
class SeedStorage:
    def __init__(self, seed=None):
        self.seed = seed
    def __call__(self):
        if self.seed is None:
            seed = None
        else:
            seed = self.seed
            self.seed += 1
        return seed

In [ ]:
MainSeedStorage = SeedStorage(42)

class MyFuncDesc(object):
    def __init__(self, func, func_name, arity, complexity):
        self.func = func
        self.func_name = func_name
        self.arity = arity
        self.complexity = complexity

class MatrixFunc(object):
    def __init__(self, func_list,
                 var_number=None, adj_matrix=None, sigma=0.0):
        self.np_norm_random_gen = np.random.RandomState(MainSeedStorage()).normal
        self.func_list = func_list
        self.sigma = sigma
        self.var_number = var_number
        self.func_number = len(func_list)
        
        if adj_matrix is None:
            self.adj_matrix = np.zeros((self.func_number, self.func_number + self.var_number),
                                       dtype=int)
        else:
            self.adj_matrix=adj_matrix
            
        self.func = None
        
    def MakeFunctionFromMatrix(self, level=0, used=set(), print_func=False):
        level_func = self.func_list[level]
        matrix_row = self.adj_matrix[level].copy()
        
        matrix_row[list(used)] = 0

        to_print = None
        rec_to_print = [''] * level_func.arity
        if print_func:
            to_print = level_func.func_name

        used.add(level)

        new_level_funcs = []
        new_level_func = None
        for i in range(level_func.arity):
            new_argmax = np.argmax(matrix_row)
            if (new_level_func is None or not matrix_row[new_argmax] == 0):
                new_level_func = new_argmax
            matrix_row[new_level_func] = 0
            # new_level_func += 1

            new_level_funcs.append(new_level_func)

            if new_level_func >= self.func_number:
                new_level_funcs[i] -= self.func_number
                rec_to_print[i] = 'x_{}'.format(new_level_funcs[i])
            else:
                new_level_funcs[i], rec_to_print[i] = \
                    self.MakeFunctionFromMatrix(new_level_funcs[i], used.copy(), print_func)

        if print_func:  
            to_print = to_print.format(*rec_to_print)

        def func(args, new_level_funcs=new_level_funcs):
            new_level_funcs_copy = new_level_funcs.copy()
            for i in range(level_func.arity):
                if not callable(new_level_funcs[i]):
                    # print(new_level_funcs[i])
                    new_level_funcs_copy[i] = args[:, new_level_funcs[i]]
                else:
                    new_level_funcs_copy[i] = new_level_funcs[i](args)
            return level_func.func(*new_level_funcs_copy)

        if level == 0:
            self.func = func
            return func, to_print
        else:
            return func, to_print

    def __call__(self, X):
        if self.func is None:
            self.MakeFunctionFromMatrix()

        if len(X.shape) == 1:
            X = X.reshape((1, X.shape[0]))
        # print(X)
        return self.func(X) + self.np_norm_random_gen(size=X.shape[0]) * self.sigma

def MakeRandomFunc(mat_func, complexity_limit=-1):
    '''
    Creates a random matrix for MatrixFunc 
    (given func_list from mat_func MatrixFunc object)
    
    Parameters
    -------------------------------------------------
    mat_func:
        MatrixFunc object with func_list in it

    complexity_limit:
        Specifies the depth of a function complexity, 
        with -1 == no limits. For example,
        sin(ln(x)) + cos(x) has a complexity of 2.
    '''
    queue = [(0, 0)]

    total_num = mat_func.func_number + mat_func.var_number
    var_set = set(range(total_num)) - set(range(mat_func.func_number))
    not_used = set(range(total_num)) - set([queue[0][0]])

    while not len(queue) == 0:
        func = mat_func.func_list[queue[0][0]]
        cmplx = queue[0][1]
        #print(queue)
        #print(cmplx, complexity_limit, func.func_name)
        chosen = []
        if (cmplx < complexity_limit or complexity_limit == -1):
            if len(not_used) >= func.arity:
                chosen = random.sample(not_used, func.arity)
            else:
                chosen = random.sample(not_used, len(not_used))

        # print(var_set)
        while len(chosen) < func.arity:
            chosen.append(random.sample(var_set, 1)[0])

        for i in range(func.arity):
            chosen_elem = chosen[i]
            # chosen_elem = random.sample(not_used, 1)[0]
            if chosen_elem < mat_func.func_number:
                not_used = not_used - set([chosen_elem])
                queue.append((chosen_elem, cmplx + mat_func.func_list[chosen_elem].complexity))
            mat_func.adj_matrix[queue[0][0], chosen_elem] = 1
        queue[:] = queue[1:]
    return mat_func

def MatrixFuncFromProbMatrix(func_list, prob_matrix, method='greedy_dfs'):
    '''
    Creates a MatrixFunc object given func_list and
    probabilities matrix prob_matrix using specified method
    
    
    Parameters
    -------------------------------------------------
    func_list:
        List of base functions to build superpositions

    prob_matrix:
        Probabilities matrix to restore a matrix defining a
        superposition
    
    method:
        A restoration method
    '''
    if method == 'greedy_dfs':
        new_matrix = np.zeros(prob_matrix.shape, dtype=int)
        def GreedyRestoreDFS(func_list, prob_matrix, new_matrix, level=0, used=set()):
            level_func = func_list[level]
            matrix_row = prob_matrix[level].copy()

            matrix_row[list(used)] = 0
            used.add(level)

            new_level_func = None
            for i in range(level_func.arity):
                new_argmax = np.argmax(matrix_row)
                if (new_level_func is None or not matrix_row[new_argmax] == 0):
                    new_level_func = new_argmax
                matrix_row[new_level_func] = 0
                # new_level_func += 1
                new_matrix[level, new_level_func] = 1

                if new_level_func < len(func_list):
                    GreedyRestoreDFS(func_list, prob_matrix, new_matrix,
                                     new_level_func, used.copy())

            return
        
        GreedyRestoreDFS(func_list, prob_matrix, new_matrix, level=0, used=set())
        
        return MatrixFunc(func_list, var_number=prob_matrix.shape[1] - prob_matrix.shape[0],
                          adj_matrix=new_matrix)

Let’s start with an example from Varfolomeeva’s work. We will create a function (define a list of functions, but for now we will not describe the matrix):

In [ ]:
f_sum = MyFuncDesc(lambda x, y, z: x + y + z, '{} + {} + {}', 3, 0)
f_times = MyFuncDesc(lambda x, y: x * y, '{} * {}', 2, 1)

eps = 0.001
f_ln = MyFuncDesc(lambda x: np.log(np.abs(x) + eps), 'ln(abs({}) + 0.001)', 1, 2)
f_sin = MyFuncDesc(np.sin, 'sin({})', 1, 1)

f_list = [f_sum, f_times, f_ln, f_sin]

MF = MatrixFunc(f_list, var_number=1)

Fill the matrix and see what function we get:

In [ ]:
MF.adj_matrix[0, 1] = 1
MF.adj_matrix[0, 2] = 1
MF.adj_matrix[1, 3] = 1
MF.adj_matrix[1, 4] = 1
MF.adj_matrix[2, 4] = 1
MF.adj_matrix[3, 4] = 1

# print(MF.adj_matrix)

f, g = MF.MakeFunctionFromMatrix(print_func=True)
print(g)

In [ ]:
X = np.linspace(1.0, 20.0, 121)
Y = MF(X.reshape(121, 1))

plt.figure(figsize=(10, 5))
plt.grid(True)
plt.plot(X, Y)

Now let’s test the random generation of functions (with a fixed list):

In [ ]:
MF = MatrixFunc(f_list, var_number=1, sigma=0.1)

MF = MakeRandomFunc(MF, 1)
# MF.adj_matrix

f, g = MF.MakeFunctionFromMatrix(print_func=True)
print(g)

X = np.linspace(0.0, 5.0, 201)
Y = MF(X.reshape(201, 1))

plt.figure(figsize=(10, 5))
plt.grid(True)
plt.plot(X, Y)

# The plan:
1. Define a set of functions ($functions_list$), and based on it generate 20–30 generating functions.
2. For each noise level (sigma), construct several noisy datasets (pairs((X,y),M), where M is the matrix that generated the function (with a fixed list).
3. Use something from sklearn, multivariate regression. Predict the matrices. Metrics compatible with sklearn are also needed here.
4. Reconstruct the matrices and compare them with the generating ones.

In [ ]:
f_sum = MyFuncDesc(lambda x1, x2, x3: x1 + x2 + x3, '{} + {} + {}', 3, 0)
f_times = MyFuncDesc(lambda x, y: x * y, '{} * {}', 2, 1)

def get_cossin_f(alpha=1.0, cos=True):
    if cos:
        return MyFuncDesc(lambda x: np.cos(alpha * x), 'cos({})'.format(str(alpha) + '({})'), 1, 1)
    else:
        return MyFuncDesc(lambda x: np.sin(alpha * x), 'sin({})'.format(str(alpha) + '({})'), 1, 1)

alphas = [1.0, 3.0, 5.0, 7.0, 9.0]

f_sin = [get_cossin_f(alpha=x, cos=False) for x in alphas]
f_cos = [get_cossin_f(alpha=x, cos=True) for x in alphas]

f_list = [f_sum, f_times, *f_sin, *f_cos]

In [ ]:
random.seed(a=472443)

MF_list = []

for i in range(0, 21):
    MF = MatrixFunc(f_list, var_number=1, sigma=0)
    MF = MakeRandomFunc(MF, 1)

    f, g = MF.MakeFunctionFromMatrix(print_func=True)
    
    flag = False
    for MF_old in MF_list:
        if (MF_old.adj_matrix == MF.adj_matrix).all():
            flag = True
            break
        
    print(g)
    if not flag:
        MF_list.append(MF)

len(MF_list)
# X = np.linspace(0.0, 5.0, 201)
# Y = MF(X.reshape(201, 1))

# plt.figure(figsize=(10, 5))
# plt.grid(True)
# plt.plot(X, Y)

In [ ]:
def mse(ground_truth, predictions):
    # print((np.linalg.norm(ground_truth - predictions, axis=1) ** 2).shape, type(predictions))
    diff = (np.linalg.norm(ground_truth - predictions, axis=1) ** 2).sum() / len(predictions)
    return diff


def do_exp_sigma_dependancy_dataset(MF_list, X=np.linspace(0.0, 5.0, 251),
                                    sigma=0.0, tries=10):
    X = X.reshape(len(X), 1)
    # sigmas = np.linspace(0.0, 3.0, 4)
    # X = np.linspace(0.0, 5.0, 251)
    data_X = np.zeros((len(MF_list) * tries, len(X)))
    data_Y = np.zeros((len(MF_list) * tries, len(np.ravel(MF_list[0].adj_matrix))))

    for i, MF in enumerate(MF_list):
        MF.sigma = sigma
        for j in range(0, tries):
            data_X[i * tries + j] = MF(X)
            data_Y[i * tries + j] = np.ravel(MF_list[i].adj_matrix)

    return data_X, data_Y


def do_exp(MF_list, f_list, sigmas):
    
    mses_before_restoration = []
    mses_after_restoration = []
    recovered = []
    
    for sigma in tqdm.tqdm_notebook(sigmas):
        data_X, data_Y = do_exp_sigma_dependancy_dataset(MF_list, sigma=sigma)

        eps = 1e-05
        non_constant = data_Y.std(axis=0) > eps

        data_Y_non_const = data_Y[:, non_constant]
        #print(data_Y_non_const.shape)

        X_train, X_test, Y_train, Y_test, Y_train_non_const, Y_test_non_const = \
            train_test_split(data_X, data_Y, data_Y_non_const, test_size=0.4, random_state=42)

        # print(Y_train_non_const.shape, Y_test_non_const.shape)

        params = {'estimator__C': np.logspace(-2, 2, 13)}

        clf = MultiOutputClassifier(LogisticRegression(penalty='l2'), n_jobs=1)



        grid_searcher = GridSearchCV(
                clf,
                params,
                n_jobs=4,
                scoring=make_scorer(mse, greater_is_better=False),
                cv=5,
                verbose=0,
                return_train_score='True'
        )

        grid_searcher = grid_searcher.fit(X_train, Y_train_non_const)

        # print(grid_searcher.best_params_['estimator__C'])
        
        clf = MultiOutputClassifier(\
            LogisticRegression(C=grid_searcher.best_params_['estimator__C'], penalty='l2'), n_jobs=4)
        clf.fit(X_train, Y_train_non_const)
        #clf.predict(X_test)[0]
        Y_test_pred = Y_test.copy()
        Y_test_pred[:, non_constant] = np.array([x[:, 1] for x in clf.predict_proba(X_test)]).T

        mses_before_restoration.append(mse(Y_test, Y_test_pred))

        new_MF_list = []
        for i in range(Y_test_pred.shape[0]):
            prob_matrix = Y_test_pred[i].reshape(MF_list[0].adj_matrix.shape)
            MF = MatrixFuncFromProbMatrix(f_list, prob_matrix, method='greedy_dfs')
            Y_test_pred[i] = np.ravel(MF.adj_matrix)

        mses_after_restoration.append(mse(Y_test, Y_test_pred))

        Y_test_pred = Y_test_pred.astype(int)

        TR_counter = 0
        for i in range(Y_test_pred.shape[0]):
            if (Y_test_pred[i] == Y_test[i]).all():
                TR_counter += 1

        recovered.append(TR_counter / Y_test_pred.shape[0])
    
    return mses_before_restoration, mses_after_restoration, recovered

In [ ]:
MainSeedStorage = SeedStorage(42)

sigmas = np.linspace(0.0, 5.0, 51)

mses_before_restoration, mses_after_restoration, recovered = do_exp(MF_list, f_list, sigmas)

In [ ]:
plt.rc('text', usetex=False)
# plt.rc('font', family='times')

csfont = {'fontname':'Times New Roman'}

fig, ax1 = plt.subplots(figsize=(14, 7))

ax1.grid(True)

color = 'darkred'
ax1.set_xlabel(u'Variance, $\sigma$', fontsize=14, fontname='Times New Roman')
ax1.set_ylabel(u'MSE', color=color, fontsize=14, fontname='Times New Roman')
ax1.plot(sigmas, mses_before_restoration, color='red', label='MSE before reconstruction')
ax1.plot(sigmas, mses_after_restoration, color='darkred', label='MSE after reconstruction')
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()  # instantiate a second axes that shares the same x-axis

color = 'darkblue'
ax2.set_ylabel(u'Proportion of correct reconstructions', color=color, fontsize=14, fontname='Times New Roman')  # we already handled the x-label with ax1
ax2.plot(sigmas, recovered, color=color, label='Proportion of correctly reconstructed superpositions')
ax2.tick_params(axis='y', labelcolor=color)


ax1.set_title(u'Dependence of MSE and the proportion of correctly reconstructed superpositions on the noise variance $\sigma$',
              fontsize=18, fontname='Times New Roman')

ax1.legend(fontsize=14, loc=(0.03, 0.7), ncol=1)
ax2.legend(fontsize=14, loc=(0.03, 0.6), ncol=1)


fig.tight_layout()  # otherwise the right y-label is slightly clipped

plt.savefig('MSE_TPR_20_func_sincos.eps', format='eps', pad_inches=0.5)

plt.show()

# Various superpositions in the test and train

In [ ]:
random.seed(a=472443)

MF_list = []

for i in range(0, 41):
    MF = MatrixFunc(f_list, var_number=1, sigma=0)
    MF = MakeRandomFunc(MF, 1)

    f, g = MF.MakeFunctionFromMatrix(print_func=True)
    
    flag = False
    for MF_old in MF_list:
        if (MF_old.adj_matrix == MF.adj_matrix).all():
            flag = True
            break
        
    # print(g)
    if not flag:
        MF_list.append(MF)

len(MF_list)

In [ ]:
def mse(ground_truth, predictions):
    # print((np.linalg.norm(ground_truth - predictions, axis=1) ** 2).shape, type(predictions))
    diff = (np.linalg.norm(ground_truth - predictions, axis=1) ** 2).sum() / len(predictions)
    return diff


def do_exp_sigma_dependancy_dataset(MF_list, X=np.linspace(0.0, 5.0, 251),
                                    sigma=0.0, tries=10):
    X = X.reshape(len(X), 1)
    # sigmas = np.linspace(0.0, 3.0, 4)
    # X = np.linspace(0.0, 5.0, 251)
    data_X = np.zeros((len(MF_list) * tries, len(X)))
    data_Y = np.zeros((len(MF_list) * tries, len(np.ravel(MF_list[0].adj_matrix))))

    for i, MF in enumerate(MF_list):
        MF.sigma = sigma
        for j in range(0, tries):
            data_X[i * tries + j] = MF(X)
            data_Y[i * tries + j] = np.ravel(MF_list[i].adj_matrix)

    return data_X, data_Y


def do_exp(MF_list, f_list, sigmas):
    
    mses_before_restoration = []
    mses_after_restoration = []
    recovered = []
    
    MF_list_train, MF_list_test = train_test_split(MF_list, test_size=0.4, random_state=42)
    
    for sigma in tqdm.tqdm_notebook(sigmas):
        
        # train_test_split(data_X, data_Y, data_Y_non_const, test_size=0.4, random_state=42)
        
        X_train, Y_train = do_exp_sigma_dependancy_dataset(MF_list_train, sigma=sigma)
        X_test, Y_test = do_exp_sigma_dependancy_dataset(MF_list_test, sigma=sigma)
        
        eps = 1e-05
        non_constant = Y_train.std(axis=0) > eps

        Y_train_non_const = Y_train[:, non_constant]
        Y_test_non_const = Y_test[:, non_constant]
        # print(data_Y_non_const.shape)
        # X_train, X_test, Y_train, Y_test, Y_train_non_const, Y_test_non_const = 
            
#         X_train, X_test, Y_train, Y_test, Y_train_non_const, Y_test_non_const = \
#             train_test_split(data_X, data_Y, data_Y_non_const, test_size=0.4, random_state=42)

        # print(Y_train_non_const.shape, Y_test_non_const.shape)

        params = {'estimator__C': np.logspace(-2, 2, 13)}

        clf = MultiOutputClassifier(LogisticRegression(penalty='l2'), n_jobs=1)



        grid_searcher = GridSearchCV(
                clf,
                params,
                n_jobs=4,
                scoring=make_scorer(mse, greater_is_better=False),
                cv=5,
                verbose=0,
                return_train_score='True'
        )

        grid_searcher = grid_searcher.fit(X_train, Y_train_non_const)

        # print(grid_searcher.best_params_['estimator__C'])
        
        clf = MultiOutputClassifier(\
            LogisticRegression(C=grid_searcher.best_params_['estimator__C'], penalty='l2'), n_jobs=4)
        clf.fit(X_train, Y_train_non_const)
        #clf.predict(X_test)[0]
        Y_test_pred = Y_test.copy()
        Y_test_pred[:, non_constant] = np.array([x[:, 1] for x in clf.predict_proba(X_test)]).T

        mses_before_restoration.append(mse(Y_test, Y_test_pred))

        new_MF_list = []
        for i in range(Y_test_pred.shape[0]):
            prob_matrix = Y_test_pred[i].reshape(MF_list[0].adj_matrix.shape)
            MF = MatrixFuncFromProbMatrix(f_list, prob_matrix, method='greedy_dfs')
            Y_test_pred[i] = np.ravel(MF.adj_matrix)

        mses_after_restoration.append(mse(Y_test, Y_test_pred))

        Y_test_pred = Y_test_pred.astype(int)

        TR_counter = 0
        for i in range(Y_test_pred.shape[0]):
            if (Y_test_pred[i] == Y_test[i]).all():
                TR_counter += 1

        recovered.append(TR_counter / Y_test_pred.shape[0])
    
    return mses_before_restoration, mses_after_restoration, recovered

In [ ]:
MainSeedStorage = SeedStorage(42)

sigmas = np.linspace(0.0, 5.0, 51)

mses_before_restoration, mses_after_restoration, recovered = do_exp(MF_list, f_list, sigmas)

In [ ]:
plt.rc('text', usetex=False)
# plt.rc('font', family='times')

csfont = {'fontname':'Times New Roman'}

fig, ax1 = plt.subplots(figsize=(14, 7))

ax1.grid(True)

color = 'darkred'
ax1.set_xlabel(u'Variance, $\sigma$', fontsize=14, fontname='Times New Roman')
ax1.set_ylabel(u'MSE', color=color, fontsize=14, fontname='Times New Roman')
ax1.plot(sigmas, mses_before_restoration, color='red', label='MSE before reconstruction')
ax1.plot(sigmas, mses_after_restoration, color='darkred', label='MSE after reconstruction')
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()  # instantiate a second axes that shares the same x-axis

color = 'darkblue'
ax2.set_ylabel(u'Ratio of the correct reconstructions', color=color, fontsize=14, fontname='Times New Roman')  # we already handled the x-label with ax1
ax2.plot(sigmas, recovered, color=color, label='Ratio of the correct reconstructions')
ax2.tick_params(axis='y', labelcolor=color)


ax1.set_title(u'Dependence of MSE and the proportion of correctly reconstructed superpositions on the noise variance $\sigma$',
              fontsize=18, fontname='Times New Roman')

ax1.legend(fontsize=14, loc=(0.03, 0.7), ncol=1)
ax2.legend(fontsize=14, loc=(0.03, 0.6), ncol=1)


fig.tight_layout()  # otherwise the right y-label is slightly clipped

plt.savefig('MSE_TPR_40_func_sincos_different_superpositions.eps', format='eps', pad_inches=0.5)

plt.show()

## Same with greater complexity

In [ ]:
random.seed(a=472443)

MF_list = []

for i in range(0, 500):
    MF = MatrixFunc(f_list, var_number=1, sigma=0)
    MF = MakeRandomFunc(MF, 2)

    f, g = MF.MakeFunctionFromMatrix(print_func=True)
    
    flag = False
    for MF_old in MF_list:
        if (MF_old.adj_matrix == MF.adj_matrix).all():
            flag = True
            break
        
    # print(g)
    if not flag:
        MF_list.append(MF)

len(MF_list)

In [ ]:
def mse(ground_truth, predictions):
    # print((np.linalg.norm(ground_truth - predictions, axis=1) ** 2).shape, type(predictions))
    diff = (np.linalg.norm(ground_truth - predictions, axis=1) ** 2).sum() / len(predictions)
    return diff


def do_exp_sigma_dependancy_dataset(MF_list, X=np.linspace(0.0, 5.0, 251),
                                    sigma=0.0, tries=10):
    X = X.reshape(len(X), 1)
    # sigmas = np.linspace(0.0, 3.0, 4)
    # X = np.linspace(0.0, 5.0, 251)
    data_X = np.zeros((len(MF_list) * tries, len(X)))
    data_Y = np.zeros((len(MF_list) * tries, len(np.ravel(MF_list[0].adj_matrix))))

    for i, MF in enumerate(MF_list):
        MF.sigma = sigma
        for j in range(0, tries):
            data_X[i * tries + j] = MF(X)
            data_Y[i * tries + j] = np.ravel(MF_list[i].adj_matrix)

    return data_X, data_Y


def do_exp(MF_list, f_list, sigmas):
    
    mses_before_restoration = []
    mses_after_restoration = []
    recovered = []
    
    MF_list_train, MF_list_test = train_test_split(MF_list, test_size=0.2, random_state=42)
    
    for sigma in tqdm.tqdm_notebook(sigmas):
        
        # train_test_split(data_X, data_Y, data_Y_non_const, test_size=0.4, random_state=42)
        
        X_train, Y_train = do_exp_sigma_dependancy_dataset(MF_list_train, sigma=sigma)
        X_test, Y_test = do_exp_sigma_dependancy_dataset(MF_list_test, sigma=sigma)
        
        eps = 1e-05
        non_constant = Y_train.std(axis=0) > eps
        
        Y_train_non_const = Y_train[:, non_constant]
        Y_test_non_const = Y_test[:, non_constant]
        print(Y_train_non_const.sum(axis=0))

        params = {'estimator__C': np.logspace(-2, 2, 13)}

        clf = MultiOutputClassifier(LogisticRegression(penalty='l2'), n_jobs=1)



        grid_searcher = GridSearchCV(
                clf,
                params,
                n_jobs=4,
                scoring=make_scorer(mse, greater_is_better=False),
                cv=5,
                verbose=0,
                return_train_score='True'
        )

        grid_searcher = grid_searcher.fit(X_train, Y_train_non_const)

        # print(grid_searcher.best_params_['estimator__C'])
        
        clf = MultiOutputClassifier(\
            LogisticRegression(C=grid_searcher.best_params_['estimator__C'], penalty='l2'), n_jobs=4)
        clf.fit(X_train, Y_train_non_const)
        #clf.predict(X_test)[0]
        Y_test_pred = Y_test.copy()
        Y_test_pred[:, non_constant] = np.array([x[:, 1] for x in clf.predict_proba(X_test)]).T

        mses_before_restoration.append(mse(Y_test, Y_test_pred))

        new_MF_list = []
        for i in range(Y_test_pred.shape[0]):
            prob_matrix = Y_test_pred[i].reshape(MF_list[0].adj_matrix.shape)
            MF = MatrixFuncFromProbMatrix(f_list, prob_matrix, method='greedy_dfs')
            Y_test_pred[i] = np.ravel(MF.adj_matrix)

        mses_after_restoration.append(mse(Y_test, Y_test_pred))

        Y_test_pred = Y_test_pred.astype(int)

        TR_counter = 0
        for i in range(Y_test_pred.shape[0]):
            if (Y_test_pred[i] == Y_test[i]).all():
                TR_counter += 1

        recovered.append(TR_counter / Y_test_pred.shape[0])
    
    return mses_before_restoration, mses_after_restoration, recovered

In [ ]:
MainSeedStorage = SeedStorage(42)

sigmas = np.linspace(0.0, 5.0, 51)

mses_before_restoration, mses_after_restoration, recovered = do_exp(MF_list, f_list, sigmas)